<a href="https://colab.research.google.com/github/Gonablitz/intern-assignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gonablitz/intern-assignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [1]:
import pandas as pd
import numpy as np

# Candidate items for a single query
candidates = pd.DataFrame({
    'item_id': ['item_A', 'item_B', 'item_C', 'item_D'],
    'raw_relevance_score': [0.42, 0.89, 0.15, 0.73]
})

# Sort by model relevance score to produce final rank
candidates['rank_position'] = candidates['raw_relevance_score'].rank(ascending=False, method='min').astype(int)
candidates = candidates.sort_values('rank_position')

print("Ranking Output Demonstration:")
print(candidates[['rank_position', 'item_id', 'raw_relevance_score']])

Ranking Output Demonstration:
   rank_position item_id  raw_relevance_score
1              1  item_B                 0.89
3              2  item_D                 0.73
0              3  item_A                 0.42
2              4  item_C                 0.15


Learning to Rank

The core goal is not to score content in isolation or assign binary tags, but to order a list of candidate items relative to each other for a given query or context. A classification model might rate two articles as relevant, but a ranking model determines which one deserves position 1 versus position 5. The output is a relative permutation of candidates that maximizes query relevance and user utility.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
def assign_ranking_target(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    conditions = [
        (df['clicked'] == 1) & (df['dwell_time_seconds'] >= 60),
        (df['clicked'] == 1) & (df['dwell_time_seconds'] >= 20) & (df['dwell_time_seconds'] < 60),
        (df['clicked'] == 1) & (df['dwell_time_seconds'] < 20),
        (df['clicked'] == 0)
    ]
    relevance_grades = [3, 2, 1, 0]
    df['relevance_grade'] = np.select(conditions, relevance_grades, default=0)
    return df

sample_impressions = pd.DataFrame({
    'query_id': ['q_10', 'q_10', 'q_10', 'q_10'],
    'item_id': ['item_A', 'item_B', 'item_C', 'item_D'],
    'clicked': [1, 1, 1, 0],
    'dwell_time_seconds': [75, 30, 8, 0]
})

print(assign_ranking_target(sample_impressions))


  query_id item_id  clicked  dwell_time_seconds  relevance_grade
0     q_10  item_A        1                  75                3
1     q_10  item_B        1                  30                2
2     q_10  item_C        1                   8                1
3     q_10  item_D        0                   0                0


Predicted Outcome: Query-relative item relevance score (graded relevance label: $y \in \{0, 1, 2, 3\}$).Label Origin: Observed user interaction signals combined into an explicit proxy rule:3 (Highly Relevant): Click + dwell time $> 60\text{s}$ OR explicit save/conversion.2 (Relevant): Click + dwell time between $20\text{s}$ and $60\text{s}$.1 (Marginal): Click + bounce ($< 20\text{s}$ dwell time).0 (Not Relevant): Impression without a click.
Moreso a pre/defined rule is a bit too static.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [5]:
from sklearn.metrics import ndcg_score

# True graded relevance vs predicted scores for a single query candidate set
y_true_relevance = np.array([[3, 2, 0, 1]]) # True relevance grades
y_predicted_scores = np.array([[0.95, 0.60, 0.10, 0.40]]) # Model predicted ranking scores

ndcg_at_3 = ndcg_score(y_true_relevance, y_predicted_scores, k=3)

print(f"Offline Metric Check:")
print(f"Calculated NDCG@3: {ndcg_at_3:.4f} (Target: >= 0.75)")

Offline Metric Check:
Calculated NDCG@3: 1.0000 (Target: >= 0.75)


Offline Metric: NDCG@k (Normalized Discounted Cumulative Gain at $k=5$ and $k=10$) along with MRR (Mean Reciprocal Rank).NDCG@5 Target: $\ge 0.75$Why: Position bias is steep—users rarely look past the top few results. NDCG heavily penalizes placing highly relevant items ($y=3$) further down the list using a logarithmic decay function $\frac{2^{rel} - 1}{\log_2(rank + 1)}$.Online Business Metric: +10% lift in Click-Through Rate at Position 1 (CTR@1) and reduction in time-to-first-click.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import pandas as pd
import duckdb

# Load starter data slice using DuckDB
try:
    df_slice = duckdb.query("""
        SELECT
            query_id,
            item_id,
            title_cosine_sim,
            bm25_text_score,
            historical_ctr,
            clicked,
            dwell_time_seconds
        FROM 'data/flyrank_starter.parquet'
        LIMIT 5
    """).df()
except Exception:
    # Fallback mockup if database file isn't loaded yet
    df_slice = pd.DataFrame({
        'query_id': ['q_401', 'q_401', 'q_401', 'q_402', 'q_402'],
        'item_id': ['doc_10', 'doc_14', 'doc_88', 'doc_03', 'doc_19'],
        'title_cosine_sim': [0.88, 0.62, 0.15, 0.91, 0.45],
        'bm25_text_score': [14.2, 9.8, 2.1, 18.5, 6.3],
        'historical_ctr': [0.12, 0.08, 0.01, 0.19, 0.04],
        'clicked': [1, 1, 0, 1, 0],
        'dwell_time_seconds': [90, 25, 0, 110, 0]
    })

# Attach target column
df_analysis = assign_ranking_target(df_slice)

print(f"Unit of Analysis Shape: {df_analysis.shape}")
print("One row = One Query-Document Candidate Pair")
df_analysis[['query_id', 'item_id', 'title_cosine_sim', 'bm25_text_score', 'relevance_grade']]

Unit of Analysis Shape: (5, 8)
One row = One Query-Document Candidate Pair


,query_id,item_id,title_cosine_sim,bm25_text_score,relevance_grade
0,q_401,doc_10,0.88,14.2,3
1,q_401,doc_14,0.62,9.8,2
2,q_401,doc_88,0.15,2.1,0
3,q_402,doc_03,0.91,18.5,3
4,q_402,doc_19,0.45,6.3,0


Unit of Analysis: One row = One Query-Document Candidate Pair within a Search or Recommendation Session

Every row pairs a specific query context with a candidate content item, containing query-document interaction features (e.g., semantic similarity, exact title match, historical CTR) and the target relevance label

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [6]:
import pandas as pd
import numpy as np

# Illustrating where fixed BM25 rule ordering fails compared to true relevance

# Check if df_analysis is defined. If not, define it using the fallback data and function.
if 'df_analysis' not in locals() and 'df_analysis' not in globals():

    # Define the assign_ranking_target function if it's not already defined
    if 'assign_ranking_target' not in globals():
        def assign_ranking_target(df: pd.DataFrame) -> pd.DataFrame:
            df = df.copy()
            conditions = [
                (df['clicked'] == 1) & (df['dwell_time_seconds'] >= 60),
                (df['clicked'] == 1) & (df['dwell_time_seconds'] >= 20) & (df['dwell_time_seconds'] < 60),
                (df['clicked'] == 1) & (df['dwell_time_seconds'] < 20),
                (df['clicked'] == 0)
            ]
            relevance_grades = [3, 2, 1, 0]
            df['relevance_grade'] = np.select(conditions, relevance_grades, default=0)
            return df

    # Create df_slice using the fallback mock-up data
    df_slice_fallback = pd.DataFrame({
        'query_id': ['q_401', 'q_401', 'q_401', 'q_402', 'q_402'],
        'item_id': ['doc_10', 'doc_14', 'doc_88', 'doc_03', 'doc_19'],
        'title_cosine_sim': [0.88, 0.62, 0.15, 0.91, 0.45],
        'bm25_text_score': [14.2, 9.8, 2.1, 18.5, 6.3],
        'historical_ctr': [0.12, 0.08, 0.01, 0.19, 0.04],
        'clicked': [1, 1, 0, 1, 0],
        'dwell_time_seconds': [90, 25, 0, 110, 0]
    })

    # Attach target column to create df_analysis
    df_analysis = assign_ranking_target(df_slice_fallback)

df_ranking_check = df_analysis.copy()

# Simple heuristic sort by BM25
df_heuristic = df_ranking_check.sort_values(by=['query_id', 'bm25_text_score'], ascending=[True, False])

print("Heuristic Ranking (BM25 only) vs True Relevance Grade:")
print(df_heuristic[['query_id', 'item_id', 'bm25_text_score', 'historical_ctr', 'relevance_grade']]) #Code corrected by Gemini


Heuristic Ranking (BM25 only) vs True Relevance Grade:
  query_id item_id  bm25_text_score  historical_ctr  relevance_grade
0    q_401  doc_10             14.2            0.12                3
1    q_401  doc_14              9.8            0.08                2
2    q_401  doc_88              2.1            0.01                0
3    q_402  doc_03             18.5            0.19                3
4    q_402  doc_19              6.3            0.04                0


Why Heuristics Fail for Ranking:
A hardcoded rule like ORDER BY bm25_text_score DESC or IF title_match THEN rank_first fails because:

Multi-Feature Interaction: Text overlap alone ignores item quality, user context, and historical popularity. A document with a slightly lower text match score but massive historical authority is usually a better rank #1 than a keyword-stuffed page.

Query Intent Diversity: Short navigational queries demand exact title matches, whereas long-tail informational queries require deep semantic understanding. Fixed rules cannot dynamically re-weight features based on query characteristics.

Pairwise Disagreements: Sorting by a single heuristic metric frequently leads to tie-breaker failures and poor top-k precision. LTR models directly optimize loss functions (e.g., LambdaMART, RankNet) that care specifically about relative candidate pairs.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.